#   QARTOD Timing QC Test
This notebook explores the functionality of QARTOD's Timing/Gap QC tests.

QARTOD supports a few different flagging routines related to time:
* `impossible_date_test`
* `data_reception_test`
* `time_gap_test`

With one additional test from the `argo` module.
* `duplicate_timestamp_test`

All of these tests are dependent on numpy's [datetime64 formatting](https://numpy.org/doc/stable/reference/arrays.scalars.html#numpy.datetime64) and return [numpy masked arrays](https://numpy.org/doc/stable/reference/maskedarray.generic.html#module-numpy.ma) of flags.

Flag definitions are described in table 2 (page 7) of the [IOOS Manual for QC of Glider Data](https://cdn.ioos.noaa.gov/media/2017/12/Manual-for-QC-of-Glider-Data_05_09_16.pdf).

*As with all data glider data analysis, glider models and operating parameters differ from model to model and keep in mind that results may vary or require experimentation.*

In [1]:
import numpy as np
import pooch
import xarray as xr
import bokeh.plotting as bk
from bokeh.models import Span, Label

#   IBM colorblind schema
color_good    = "#648FFF"
color_suspect = "#FFB000"
color_bad     = "#DC267F"

from ioos_qc import qartod, argo

bk.output_notebook()

Loading BokehJS ...

In [2]:
url = "https://github.com/ioos/ioos_qc/releases/download"
version = "2.1.0"
fname = "nrt_SEA067_M37.nc"

download = pooch.retrieve(
    url=f"{url}/{version}/{fname}",
    known_hash="sha256:06e8a79cc17a2d55bb32dbfdc85f9922c1a1cc14726df004ae971125f91b27ac",
)
ds = xr.open_dataset(download)

In [3]:
test_data = ds.coords["time"].to_numpy()
test_data

array(['2022-11-13T08:54:01.331000064', '2022-11-13T08:54:31.328999936',
       '2022-11-13T08:55:01.376000000', ...,
       '2022-11-27T10:08:03.711000064', '2022-11-27T10:08:33.729999872',
       '2022-11-27T10:08:48.020999936'],
      shape=(7722,), dtype='datetime64[ns]')

##  Impossible date test
The impossible date test exists to confirm that provided timestamps are:
1. Possible ("2021-01-40" makes no sense).
2. Not in the future.
3. (optional) Within a designated window.

Item 1 above is handled by the np.datetime64 library and errors out if the glider timestamp cannot be parsed. This includes things like leap-year checks and HH:MM:SS formatting.

Any timestamps found to violate rules 2 or 3 are flagged as 4 (FAIL). Entries in the time array that are not a time "NaT" are flagged as 9 (MISSING).

In [4]:
#   Check for any impossible dates, i.e, any NaTs or future points, without a specified window to flag.
impossible_flags = qartod.impossible_date_test(test_data)
any(impossible_flags == 4)

False

Since this data has no flags, this example will introduce some.
* A few NaTs
* Define the window to be within some of the data

In [5]:
impossible_test_data = test_data[2325:2375].copy()
impossible_test_data[16:18] = np.datetime64("NaT")
impossible_test_data[31] = np.datetime64("NaT")

lower_bound = 40
window = (impossible_test_data[lower_bound], np.datetime64("NOW"))

impossible_flags = qartod.impossible_date_test(impossible_test_data, window)
print(f"Data is NaT at: {np.where(impossible_flags==9)[0]}")
print(f"Data is flagged as bad at: {np.where(impossible_flags == 4)[0]}")

Data is NaT at: [16 17 31]
Data is flagged as bad at: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 18 19 20 21 22 23 24 25
 26 27 28 29 30 32 33 34 35 36 37 38 39]


In [6]:
good = (impossible_flags == 1)
bad  = (impossible_flags == 4)
p = bk.figure(
    title="Impossible date test",
    y_axis_type="datetime", y_axis_label="Timestamp",
    x_axis_label="Index", height=350, width=500, min_border=20)

p.scatter(
    np.arange(len(impossible_test_data))[good], impossible_test_data[good],
    color=color_good,
    legend_label="Within range (GOOD)")
p.scatter(
    np.arange(len(impossible_test_data))[bad], impossible_test_data[bad],
    color=color_bad,
    legend_label="Outside range (FAIL)")

for idx in np.flatnonzero(impossible_flags == 9):
    p.add_layout(
        Span(
            location = idx,
            dimension = "height",
            line_color = color_bad,
            line_width = 2,
        )
    )

p.add_layout(Span(location=lower_bound, dimension="height", line_color="black", line_dash="dotted"))
text = Label(x=18, y=impossible_test_data[-2], text="NaT", text_font_size="8pt")
p.add_layout(text)
p.title.align = "center"
p.legend.label_text_font_size = "8pt"
p.legend.location = "bottom_right"
bk.show(p)

## Data reception test
The data reception test checks for data that has been received within a certain timeframe in hours. It does this by comparing each timestamp against either the specified `from_time` value, or the current time (default).

This test is more useful on flagging streams or near real-time quality control, less so on complete datasets. 

In [7]:
#   Specify a cutoff for flagging data 24 hours from 2022-11-29 (the data begins on the 13th)
start_window = "2022-11-28T00:00:00.000"
reception_flags = qartod.data_reception_test(test_data, fail_span=24, from_time=start_window)
interest_value = np.where(reception_flags==4)[0][-1]
print(f"First point more than 24 hours away at ix={interest_value}: {test_data[interest_value]}")

First point more than 24 hours away at ix=7148: 2022-11-26T22:58:08.023000064


In [8]:
good = (reception_flags == 1)
bad  = (reception_flags == 4)
p = bk.figure(
    title="Data reception test",
    y_axis_type="datetime", y_axis_label="Timestamp",
    x_axis_label="Index", height=350, width=500, min_border=20)

p.scatter(
    np.arange(len(test_data))[good], test_data[good],
    color=color_good, legend_label="Within range (GOOD)")
p.scatter(
    np.arange(len(test_data))[bad], test_data[bad],
    color=color_bad, legend_label="Outside range (FAIL)")

p.add_layout(Span(location=test_data[interest_value], dimension="width", line_color="black", line_dash="dotted"))
p.add_layout(Span(location=np.datetime64(start_window), dimension="width", line_color="black", line_dash="dotted"))
p.title.align = "center"
p.legend.label_text_font_size = "8pt"
p.legend.location = "bottom_right"
p.y_range.start = test_data[6000]
p.y_range.end   = np.datetime64("2022-11-29T00:00:00.000")
p.x_range.start = 6000
p.x_range.end   = len(test_data)+50
bk.show(p)

##  Time gap test
The time gap test looks at the reporting frequency of the data and flags points that are identified as gaps of a certain duration in hours.

It flags the point following the gap, rather than the point preceeding the gap.

This test is best for checking on final delayed-mode data, or leniently with NRT data.

In [9]:
#   Flag gaps using the default 2 hour window
gap_flags = qartod.time_gap_test(test_data)
print(f"Data is flagged as bad at: {np.where(gap_flags == 4)[0]}")
print(f"Number of potential data gaps: {len(np.where(gap_flags == 4)[0])}")

Data is flagged as bad at: [ 466  602  727  798  868  945 1040 1132 1223 1306 1432 1507 1688 1765
 1847 1932 2028 2318 2385 2509 2574 2635 2699 2769 2840 2907 2964 3029
 3074 3124 3180 3236 3287 3343 3403 3471 3539 3617 3701 3787 3878 3965
 4044 4121 4191 4266 4339 4411 4488 4574 4661 4739 5032 5096 5158 5225
 5289 5353 5421 5491 5561 5626 5677 5744 5790 5851 5900 5957 6018 6075
 6121 6177 6243 6303 6342 6403 6464 6523 6573 6624 6675 6723 6767 6817
 6864 6914 6962 7002]
Number of potential data gaps: 88


In [10]:
good = (gap_flags == 1)
bad  = (gap_flags == 4)
p = bk.figure(
    title="Data gap test",
    y_axis_type="datetime", y_axis_label="Timestamp",
    x_axis_label="Index", height=350, width=500, min_border=20)

p.scatter(
    np.arange(len(test_data))[good], test_data[good],
    color=color_good, legend_label="Within range (GOOD)")
p.scatter(
    np.arange(len(test_data))[bad], test_data[bad],
    color=color_bad, legend_label="Outside range (FAIL)",
    size=6)

p.title.align = "center"
p.legend.label_text_font_size = "8pt"
p.legend.location = "bottom_right"
p.y_range.start = test_data[500]
p.y_range.end   = np.datetime64("2022-11-16T00:00:00.000")
p.x_range.start = 500
p.x_range.end   = 1900
bk.show(p)

By nature of NRT data, the default threshold isn't always the best metric for continuous data and many "gaps" can be expected. Note in the figure above, however, that around indices 1375 and 1650 there is are small gaps that are within the 2-hour window of the previous point and are not flagged.

Assigning to something more lenient, like 4 hours, will give different results.

In [11]:
gap_flags = qartod.time_gap_test(test_data, fail_span=4)
print(f"Data is flagged as bad at: {np.where(gap_flags == 4)[0]}")
print(f"Number of potential data gaps: {len(np.where(gap_flags == 4)[0])}")

Data is flagged as bad at: [1932 3403]
Number of potential data gaps: 2


In [12]:
good = (gap_flags == 1)
bad  = (gap_flags == 4)
p = bk.figure(
    title="Data gap test",
    y_axis_type="datetime", y_axis_label="Timestamp",
    x_axis_label="Index", height=350, width=500, min_border=20)

p.scatter(
    np.arange(len(test_data))[good], test_data[good],
    color=color_good, legend_label="Within range (GOOD)")
p.scatter(
    np.arange(len(test_data))[bad], test_data[bad],
    color=color_bad, legend_label="Outside range (FAIL)",
    size=6)

p.title.align = "center"
p.legend.label_text_font_size = "8pt"
p.legend.location = "bottom_right"
p.y_range.start = test_data[1800]
p.y_range.end   = np.datetime64("2022-11-20T00:00:00.000")
p.x_range.start = 1800
p.x_range.end   = 4000
bk.show(p)

##  Duplicate timestamp test

In addition, there is a flagging test in `argo.py` for duplicate timestamps: `duplicate_timestamp_test`.

This function only needs the time array. This should ideally be in the `np.datetime64` format.

These data are flagged as QARTOD "SUSPECT", or flag 3.

In [13]:
duplicate_data = np.append(test_data, test_data[0]) #   Add a duplicate timestamp to the very end of the data.
duplicate_flags= argo.duplicate_timestamp_test(duplicate_data)

In [14]:
np.where(duplicate_flags.data == 3)[0]

array([   0, 7722])

In [15]:
good    = (duplicate_flags == 1)
suspect = (duplicate_flags == 3)
p = bk.figure(
    title="Duplicate timestamp test",
    y_axis_type="datetime", y_axis_label="Timestamp",
    x_axis_label="Index", height=350, width=500, min_border=20)

p.scatter(
    np.arange(len(duplicate_data))[good], duplicate_data[good],
    color=color_good, legend_label="Unique (GOOD)")
p.scatter(
    np.arange(len(duplicate_data))[suspect], duplicate_data[suspect],
    color=color_suspect, legend_label="Identical (SUSPECT)",
    size=6)

p.title.align = "center"
p.legend.label_text_font_size = "8pt"
p.legend.location = "top_left"

bk.show(p)